# Preprocess LINCS

This notebook takes LDS-1481 data and extracts relevant dataframes for downstream analysis

# Imports

In [2]:
import numpy as np
import networkx as nx
import os
import pandas as pd
from cmapPy.pandasGEXpress import parse
import pickle
import warnings

# Directories

In [3]:
# Define current directory
cwd = os.getcwd()
# Define root directory
root = os.path.dirname(os.path.dirname(cwd))

# Define DATA directory
#DATA = cwd.parents[1]/'data'/'canada'
DATA = root + '/data/canada/'

# Define INPUT directory
INPUT = DATA + 'input/'
# Define LINCS directory
LINCS = INPUT + 'LDS-1481/'

# Define OUTPUT directory
OUTPUT = DATA + 'output/'

# Functions

## General

In [18]:
def list_from_file(path):
    '''
    Converts a .txt file to a list
    '''

    with open(f'{path}', 'r', encoding = 'utf-8') as f:
        list_file = [line.strip() for line in f]
    
    return list_file

def list_to_file(path, data):
      '''
      Saves a list or set to a .txt file with no header.
      '''

      with open(path, 'w') as f:
            for item in sorted(data):
                  f.write(f'{item}\n')

def pickle_load(path: str, report: bool = False):
    '''
    Loads pickled data.
    '''

    with open(path, 'rb') as f:
        data = pickle.load(f)

    if report == True:
        if type(data) == nx.Graph:
            num_nodes = len(data.nodes)
            num_edges = len(data.edges)
            print('>> pickle_load')
            print(f'Pickled graph loaded w/ {num_nodes:,} nodes and {num_edges:,} edges')
            print()
        else:
            print('>> pickle_load')
            print(f'Pickled file loaded')
            print()

    return data

def pickle_save(path: str, data, report: bool = False):
    '''
    Pickles data.
    '''

    with open(path, 'wb') as f:
        pickle.dump(data, f)

    if report == True:
        if type(data) == nx.Graph:
            num_nodes = len(data.nodes)
            num_edges = len(data.edges)
            print('>> pickle_save')
            print(f'Graph w/ {num_nodes:,} nodes and {num_edges:,} edges pickled')
        else:
            print('>> pickle_save')
            print(f'Data pickled')
            print()

# LINCS

## Filters

In [46]:
# Define filters
FILTER_CELLS = ['HT29']
FILTER_TIMEPOINTS = ['6']
FILTER_DOSES = ['10.0']

## Signature Info

Formats LINCS signature information for later use.

In [67]:
# Load data
df_signature_info = pd.read_csv(LINCS + 'GSE92742_Broad_LINCS_sig_info.txt.gz', compression = 'gzip', sep = '\t', dtype = str)
# Rename columns
df_signature_info.rename(columns = {'pert_id' : 'perturbagen_id', 'pert_iname' : 'perturbagen_name'}, inplace = True)
# Save data
df_signature_info.to_csv(OUTPUT + 'df_signature_info.csv', index = False)
# Show data
df_signature_info.head()

,sig_id,perturbagen_id,perturbagen_name,pert_type,cell_id,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_time_unit,pert_itime,distil_id
0,AML001_CD34_24H:A05,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X1_F1B10:A05
1,AML001_CD34_24H:A06,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X3_F1B10:A06
2,AML001_CD34_24H:B05,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X1_F1B10:B05|AML001_CD34_24H_X...
3,AML001_CD34_24H:B06,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X3_F1B10:B06
4,AML001_CD34_24H:BRD-A03772856:0.37037,BRD-A03772856,BRD-A03772856,trt_cp,CD34,0.37037,µM,500 nM,24,h,24 h,AML001_CD34_24H_X1_F1B10:J04|AML001_CD34_24H_X...


## Perturbagen Info

Takes **Signature Info** and extracts list of unique perturbagen IDs and their names.

In [10]:
# Load data
df_perturbagen_info = pd.read_csv(OUTPUT + 'df_signature_info.csv', dtype = str)
# Extract relevant data
df_perturbagen_info = df_perturbagen_info[['perturbagen_id', 'perturbagen_name']].copy()
# Remove duplicates
df_perturbagen_info.drop_duplicates(inplace = True, ignore_index = True)
# Save data
df_perturbagen_info.to_csv(OUTPUT + 'df_perturbagen_info.csv', index = False)
# Show data
df_perturbagen_info.head()

,perturbagen_id,perturbagen_name
0,DMSO,DMSO
1,BRD-A03772856,BRD-A03772856
2,BRD-A19037878,trichostatin-a
3,BRD-A19500257,geldanamycin
4,BRD-A34037822,KUC107191N


### Statins

Extracts perturbagen IDs for -statin compounds, used as reference perturbagens in downstream analysis.

In [14]:
# Filter data
df_statins = df_perturbagen_info[df_perturbagen_info['perturbagen_name'].str.contains('statin')]
# Get unique statin names
list_statins = list(pd.unique(df_statins['perturbagen_name']))
# Get unique statin IDs
list_statin_ids = list(pd.unique(df_statins['perturbagen_id']))
# Get number of statins
num_statins = len(list_statins)
# Save data
list_to_file(OUTPUT + 'list_statin_ids.txt', list_statin_ids)
# Report
print(f'{num_statins} unique statins found')
print(list_statins)
# Show data
df_statins.head()

22 unique statins found
['trichostatin-a', 'cilastatin', 'mevastatin', 'wiskostatin', 'pravastatin', 'necrostatin-1', 'lovastatin', 'simvastatin', 'alrestatin', 'dephostatin', 'fluvastatin', 'rosuvastatin', 'pepstatin', 'blebbistatin', 'fatostatin', 'atorvastatin', 'pitavastatin', 'cerivastatin', 'somatostatin', 'tubastatin-a', 'erbstatin-analog', 'nystatin']


,perturbagen_id,perturbagen_name
2,BRD-A19037878,trichostatin-a
4532,BRD-A90311807,cilastatin
4781,BRD-K94441233,mevastatin
4810,BRD-A18579359,wiskostatin
5048,BRD-K60511616,pravastatin


### Profens

Extracts perturbagen IDs for -profen compounds, used as reference perturbagens in downstream analysis.

In [15]:
# Filter data
df_profens = df_perturbagen_info[df_perturbagen_info['perturbagen_name'].str.contains('profen')]
# Get unique profen names
list_profens = list(pd.unique(df_profens['perturbagen_name']))
# Get unique profen IDs
list_profen_ids = list(pd.unique(df_profens['perturbagen_id']))
# Get number of profens
num_profens = len(list_profens)
# Save data
list_to_file(OUTPUT + 'list_profen_ids.txt', list_profen_ids)
# Report
print(f'{num_profens} unique profens found')
print(list_profens)
# Show data
df_profens.head()

13 unique profens found
['carprofen', 'ketoprofen', 'ibuprofen-(S)', 'profenamine', 'ibuprofen', 'indoprofen', 'tiaprofenic-acid', 'loxoprofen', 'flurbiprofen-(+/-)', 'dexketoprofen', 'pranoprofen', 'fenoprofen', 'suprofen']


,perturbagen_id,perturbagen_name
5181,BRD-A17411484,carprofen
5249,BRD-A97739905,ketoprofen
5317,BRD-K14965640,ibuprofen-(S)
5892,BRD-A16311756,profenamine
5895,BRD-A17655518,ibuprofen


### Opioids

Imports list of opioid compound names and extracts any associated perturbagen IDs in LINCS data.

In [23]:
# Load data
list_opioids = list_from_file(INPUT + 'list_opioids.txt')
# Format list
list_opioids = [entry.lower() for entry in list_opioids]
# Get number of opioids
num_opioids = len(pd.unique(list_opioids))
# Filter perturbagen data
df_opioid = df_perturbagen_info[df_perturbagen_info['perturbagen_name'].isin(list_opioids)]
# Get unique perturbagen IDs
list_opioid_ids = list(pd.unique(df_opioid['perturbagen_id']))
# Get number of unique IDs found
num_ids = len(list_opioid_ids)
# Save data
list_to_file(OUTPUT + 'list_opioid_ids.txt', list_opioid_ids)

# Report
percent_opioid = num_ids / num_opioids * 100
print(f'{percent_opioid:.2f}% of opioids found in perturbagen data ({num_ids}/{num_opioids})')
# Show data
df_opioid.head()

21.05% of opioids found in perturbagen data (12/57)


,perturbagen_id,perturbagen_name
4866,BRD-A92651262,nalbuphine
5165,BRD-A02710418,meptazinol
5374,BRD-K33211335,dextromethorphan
5413,BRD-K51033547,tramadol
5604,BRD-A66559694,naltrexone


### HDAC Inhibitors

Imports list of hdac inhibitor compound names and extracts any associated perturbagen IDs in LINCS data.

In [24]:
# Load data
list_hdacs = list_from_file(INPUT + 'list_hdac.txt')
# Format list
list_hdacs = [entry.lower() for entry in list_hdacs]
# Get number of hdacs
num_hdacs = len(pd.unique(list_hdacs))
# Filter perturbagen data
df_hdac = df_perturbagen_info[df_perturbagen_info['perturbagen_name'].isin(list_hdacs)]
# Get unique perturbagen IDs
list_hdac_ids = list(pd.unique(df_hdac['perturbagen_id']))
# Get number of unique IDs found
num_ids = len(list_hdac_ids)
# Save data
list_to_file(OUTPUT + 'list_hdac_ids.txt', list_hdac_ids)

# Report
percent_hdac = num_ids / num_hdacs * 100
print(f'{percent_hdac:.2f}% of hdacs found in perturbagen data ({num_ids}/{num_hdacs})')
# Show data
df_hdac.head()

33.33% of hdacs found in perturbagen data (9/27)


,perturbagen_id,perturbagen_name
56,BRD-K81418486,vorinostat
5343,BRD-K22503835,scriptaid
8107,BRD-K02130563,panobinostat
8142,BRD-K13810148,givinostat
8480,BRD-K17743125,belinostat


## Gene Info

Formats gene ID, symbol, name and landmark information.

In [69]:
# Load data
df_gene_info = pd.read_csv(LINCS + 'GSE92742_Broad_LINCS_gene_info.txt.gz', compression = 'gzip', sep = '\t')

# Define rename map
dict_rename = {
    'pr_gene_id' : 'rid',
    'pr_gene_symbol' : 'lincs_name',
    'pr_gene_title' : 'lincs_desc',
    'pr_is_lm' : 'landmark',
    'pr_is_bing' : 'inferred'
}
# Rename columns
df_gene_info.rename(columns = dict_rename, inplace = True)

# Save data
df_gene_info.to_csv(OUTPUT + 'df_gene_info.csv', index = False)
# Show data
df_gene_info.head()

,rid,lincs_name,lincs_desc,landmark,inferred
0,780,DDR1,discoidin domain receptor tyrosine kinase 1,1,1
1,7849,PAX8,paired box 8,1,1
2,2978,GUCA1A,guanylate cyclase activator 1A,0,0
3,2049,EPHB3,EPH receptor B3,0,1
4,2101,ESRRA,estrogen related receptor alpha,0,1


## Landmark Info

Takes **Gene Info** and extracts landmark only data, and a list of landmark gene names.

In [70]:
# Load data
df_landmark_info = pd.read_csv(OUTPUT + 'df_gene_info.csv')
# Filter data
df_landmark_info = df_landmark_info[df_landmark_info['landmark'] == 1]
# Drop columns
df_landmark_info.drop(columns = ['landmark', 'inferred'], inplace = True)
# Save data
df_landmark_info.to_csv(OUTPUT + 'df_landmark_info.csv', index = False)

# Extract landmark gene list
list_landmark = list(pd.unique(df_landmark_info['lincs_name']))
# Save data
list_to_file(OUTPUT + 'list_landmark.txt', list_landmark)

# Report
print(f'{len(list_landmark):,} unique landmark genes extracted from LINCS data')
# Show data
df_landmark_info.head()

978 unique landmark genes extracted from LINCS data


,rid,lincs_name,lincs_desc
0,780,DDR1,discoidin domain receptor tyrosine kinase 1
1,7849,PAX8,paired box 8
25,6193,RPS5,ribosomal protein S5
43,23,ABCF1,ATP binding cassette subfamily F member 1
49,9552,SPAG7,sperm associated antigen 7


## Signature Filter IDs

Takes signature info and extracts signature IDs found using defined **Filters**.

In [71]:
# Load data
df_signature_info = pd.read_csv(OUTPUT + 'df_signature_info.csv', dtype = str)

# Filter data
df_filter_ids = df_signature_info[(df_signature_info['cell_id'].isin(FILTER_CELLS)) &
                                  (df_signature_info['pert_time'].isin(FILTER_TIMEPOINTS)) &
                                  (df_signature_info['pert_dose'].isin(FILTER_DOSES))]

# Extract signature IDs
list_filter_ids = pd.unique(df_filter_ids['sig_id'])
# Save data
list_to_file(OUTPUT + 'list_filter_ids.txt', list_filter_ids)

# Report
print('Signature ID Filters:')
print(f'> Cell(s): {FILTER_CELLS}')
print(f'> Timepoint(s): {FILTER_TIMEPOINTS}')
print(f'> Dose(s): {FILTER_DOSES}')
print()
num_perturbagens = len(pd.unique(df_filter_ids['perturbagen_id']))
print(f'{num_perturbagens:,} unique perturbagens found after filtering')

# Show data
df_filter_ids.head()

Signature ID Filters:
> Cell(s): ['HT29']
> Timepoint(s): ['6']
> Dose(s): ['10.0']

5,045 unique perturbagens found after filtering


,sig_id,perturbagen_id,perturbagen_name,pert_type,cell_id,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_time_unit,pert_itime,distil_id
50538,CPC004_HT29_6H:BRD-A00546892-001-01-8:10,BRD-A00546892,biperiden,trt_cp,HT29,10.0,µM,10 µM,6,h,6 h,CPC004_HT29_6H_X1_B3_DUO52HI53LO:J19|CPC004_HT...
50539,CPC004_HT29_6H:BRD-A00993607-003-15-4:10,BRD-A00993607,alprenolol,trt_cp,HT29,10.0,µM,10 µM,6,h,6 h,CPC004_HT29_6H_X1_B3_DUO52HI53LO:E24|CPC004_HT...
50540,CPC004_HT29_6H:BRD-A01593789-001-02-3:10,BRD-A01593789,chlormadinone-acetate,trt_cp,HT29,10.0,µM,10 µM,6,h,6 h,CPC004_HT29_6H_X1_B3_DUO52HI53LO:H21|CPC004_HT...
50541,CPC004_HT29_6H:BRD-A01643550-001-03-1:10,BRD-A01643550,prednisolone-acetate,trt_cp,HT29,10.0,µM,10 µM,6,h,6 h,CPC004_HT29_6H_X1_B3_DUO52HI53LO:P13|CPC004_HT...
50542,CPC004_HT29_6H:BRD-A02006392-001-09-9:10,BRD-A02006392,nitrendipine,trt_cp,HT29,10.0,µM,10 µM,6,h,6 h,CPC004_HT29_6H_X1_B3_DUO52HI53LO:F14|CPC004_HT...


## Signature Filter Data

Takes **Signature Filter IDs**, **Gene Info** and **Perturbagen Info** and extracts LINCS Level 5 values and associated metadata for those **Signature Filter IDs**.

### LINCS Filter

Takes **Signature Filter IDs** and filters LINCS data.

In [6]:
warnings.filterwarnings('ignore')

# Load data
list_filter_ids = list_from_file(OUTPUT + 'list_filter_ids.txt')

# Filter LINCS data
df_filter_lincs = parse.parse(LINCS + 'LDS-1481_1.0.gctx', cid = list_filter_ids).data_df

# Extract column IDs for metadata
list_metadata_columns = list(df_filter_lincs.columns)
# Save data
list_to_file(OUTPUT + 'list_metadata_columns.txt', list_metadata_columns)

# Show data
df_filter_lincs.head()

cid,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,CPC005_HT29_6H:BRD-A07824748-001-02-6:10,CPC004_HT29_6H:BRD-K20482099-001-01-1:10,CPC005_HT29_6H:BRD-K62929068-001-03-3:10,CPC005_HT29_6H:BRD-K43405658-001-01-8:10,CPC004_HT29_6H:BRD-K03670461-001-02-0:10,CPC004_HT29_6H:BRD-K36737713-001-01-6:10,CPC005_HT29_6H:BRD-K51223576-001-01-3:10,CPC004_HT29_6H:BRD-A14966924-001-03-0:10,CPC004_HT29_6H:BRD-K79131256-001-08-8:10,...,PCLB001_HT29_6H:BRD-K45028728-001-02-0:10,PCLB001_HT29_6H:BRD-A15100685-001-02-6:10,PCLB001_HT29_6H:BRD-A33746814-001-01-0:10,PCLB001_HT29_6H:BRD-K07572174-001-22-0:10,PCLB001_HT29_6H:BRD-A70150975-001-01-4:10,PCLB001_HT29_6H:BRD-A69592287-001-01-3:10,PCLB001_HT29_6H:BRD-A58955223-001-02-0:10,PCLB001_HT29_6H:BRD-K36452089-001-01-8:10,PCLB001_HT29_6H:BRD-A34462049-001-04-4:10,PCLB001_HT29_6H:BRD-A19037878-001-02-3:10
rid,,,,,,,,,,,,,,,,,,,,,
5720,-2.999525,-0.623000,0.141888,-0.752267,0.380861,0.076565,-0.749507,-0.284185,0.664252,-0.007322,...,-0.227952,1.056974,-1.633453,-0.368609,-0.941537,-2.133886,0.196782,0.700733,-0.349926,-0.032394
466,-1.922989,-1.483600,-0.004969,-1.476633,0.828392,0.913175,-0.654432,0.067755,0.684563,1.565995,...,-0.956585,0.259887,-2.375699,-0.220565,-1.854915,-1.416142,0.007847,0.721920,0.123217,-3.753825
6009,-0.742801,-0.406733,-0.614206,-3.505600,0.381193,0.356511,-0.324873,-0.300248,0.563278,1.232050,...,0.776871,-0.175295,-2.321007,-0.337826,-0.622213,-1.499496,0.309883,-0.555653,0.591362,-1.339114
2309,-1.567428,0.373000,0.258819,0.750733,-0.013309,-0.172938,-0.571196,0.091755,0.042603,-1.011316,...,-0.087529,-0.548147,-1.061960,1.320128,-0.104126,-0.584865,1.146974,-0.014965,2.970267,-0.826604
387,-2.488176,-0.959167,0.737944,-1.150100,-0.292709,-1.357093,0.885794,-0.627101,-0.029998,-0.127312,...,-0.123872,0.310397,-1.934787,-0.110689,-0.655762,-1.680875,0.470595,0.158953,-0.369358,0.116622


### Signature Metadata

Takes **LINCS Filter** data, extracts signature metadata from column IDs and merges with **Perturbagen Data**.

In [7]:
# Load data
list_metadata_columns = list_from_file(OUTPUT + 'list_metadata_columns.txt')
df_perturbagen_info = pd.read_csv(OUTPUT + 'df_perturbagen_info.csv')

# Convert to dataframe
df_signature_metadata = pd.DataFrame(list_metadata_columns, columns = ['cid'])

# Copy column
df_signature_metadata['metadata'] = df_signature_metadata['cid']
# Extract perturbagen ID
df_signature_metadata['metadata'] = df_signature_metadata['metadata'].str.replace(r'-(?:[^-]+-){2}[^-]+(?=:\d+$)', '', regex = True)
# Split metadata column
df_signature_metadata[['data', 'perturbagen_id', 'dose']] = df_signature_metadata['metadata'].str.split(':', expand = True)
# Split data column
df_signature_metadata[['dataset', 'cell_line', 'timepoint']] = df_signature_metadata['data'].str.split('_', expand = True)
# Drop columns
df_signature_metadata.drop(columns = ['metadata', 'data'], inplace = True)
# Merge with perturbagen data
df_signature_metadata = pd.merge(df_signature_metadata, df_perturbagen_info, how = 'left', on  = 'perturbagen_id')

# Save data
df_signature_metadata.to_csv(OUTPUT + 'df_signature_metadata.csv', index = False)
# Show data
df_signature_metadata.head()

,cid,perturbagen_id,dose,dataset,cell_line,timepoint,perturbagen_name
0,CPC004_HT29_6H:BRD-A00546892-001-01-8:10,BRD-A00546892,10,CPC004,HT29,6H,biperiden
1,CPC004_HT29_6H:BRD-A00993607-003-15-4:10,BRD-A00993607,10,CPC004,HT29,6H,alprenolol
2,CPC004_HT29_6H:BRD-A01593789-001-02-3:10,BRD-A01593789,10,CPC004,HT29,6H,chlormadinone-acetate
3,CPC004_HT29_6H:BRD-A01643550-001-03-1:10,BRD-A01643550,10,CPC004,HT29,6H,prednisolone-acetate
4,CPC004_HT29_6H:BRD-A02006392-001-09-9:10,BRD-A02006392,10,CPC004,HT29,6H,nitrendipine


### Signature Values

Merges **LINCS Filter**, **Gene Info** and **Signature Metadata**.

In [32]:
# Load data
df_gene_info = pd.read_csv(OUTPUT + 'df_gene_info.csv')
df_signature_metadata = pd.read_csv(OUTPUT + 'df_signature_metadata.csv')

# Copy data
df_signature_values = df_filter_lincs.copy()
# Reset index
df_signature_values.reset_index(inplace = True)
# Assert datatyes
df_signature_values['rid'] = df_signature_values['rid'].astype(int)
df_gene_info['rid'] = df_gene_info['rid'].astype(int)

# Define merge columns
list_info_columns = ['rid', 'lincs_name', 'lincs_desc', 'landmark']
# Merge data
df_signature_values = pd.merge(df_signature_values, df_gene_info[list_info_columns], how = 'left', on = 'rid')
# Melt on expression values
df_signature_values = pd.melt(df_signature_values, id_vars = list_info_columns, var_name = 'cid', value_name = 'dexp')

# Merge with metadata
df_signature_values = pd.merge(df_signature_values, df_signature_metadata, how = 'left', on  = 'cid')
# Define column list
list_columns = ['cid', 'dataset', 'cell_line', 'perturbagen_id', 'perturbagen_name', 'dose', 'timepoint', 'rid', 'lincs_name', 'lincs_desc', 'landmark', 'dexp']
# Rename column
df_signature_values.rename(columns = {'rid' : 'gene_id'}, inplace = True)

# Pickle data
df_signature_values.to_parquet(OUTPUT + 'df_signature_values.parquet', compression = 'zstd')
#pickle_save(OUTPUT + 'df_signature_values.pkl', df_signature_values)
# Show data
df_signature_values.head()

,gene_id,lincs_name,lincs_desc,landmark,cid,dexp,perturbagen_id,dose,dataset,cell_line,timepoint,perturbagen_name
0,5720,PSME1,proteasome activator subunit 1,1,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,-2.999525,BRD-A85280935,10,CPC005,HT29,6H,quinpirole
1,466,ATF1,activating transcription factor 1,1,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,-1.922989,BRD-A85280935,10,CPC005,HT29,6H,quinpirole
2,6009,RHEB,Ras homolog enriched in brain,1,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,-0.742801,BRD-A85280935,10,CPC005,HT29,6H,quinpirole
3,2309,FOXO3,forkhead box O3,1,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,-1.567428,BRD-A85280935,10,CPC005,HT29,6H,quinpirole
4,387,RHOA,ras homolog family member A,1,CPC005_HT29_6H:BRD-A85280935-003-01-7:10,-2.488176,BRD-A85280935,10,CPC005,HT29,6H,quinpirole
